In [1]:
import sys
sys.path.append("../src")

import pandas as pd
from cfb_upsets.features import add_features

df = pd.read_csv("../data/processed/games_clean_all.csv")
df = add_features(df)
df_model = df.dropna(subset=["upset"]).copy()

In [2]:
from scipy.stats import chi2_contingency
import pandas as pd

# Build a contingency table: rows = conference, columns = upset (0/1) counts
contingency = pd.crosstab(df_model["underdog_conference"], df_model["upset"])
print(contingency)

chi2, p_value, dof, expected = chi2_contingency(contingency)

print(f"\nChi-square statistic: {chi2:.3f}")
print(f"Degrees of freedom: {dof}")
print(f"P-value: {p_value:.4f}")

upset                0.0  1.0
underdog_conference          
ACC                  291  135
American Athletic    306   91
Big 12               239  117
Big Ten              318  106
Conference USA       285  103
FBS Independents     102   34
Mid-American         314  108
Mountain West        285   95
Pac-12               178   55
SEC                  261  105
Sun Belt             284  130

Chi-square statistic: 22.182
Degrees of freedom: 10
P-value: 0.0142


In [3]:
contingency_home = pd.crosstab(df_model["is_home_underdog"], df_model["upset"])
print(contingency_home)

chi2_home, p_value_home, dof_home, expected_home = chi2_contingency(contingency_home)

print(f"\nChi-square statistic: {chi2_home:.3f}")
print(f"Degrees of freedom: {dof_home}")
print(f"P-value: {p_value_home:.4f}")

upset              0.0  1.0
is_home_underdog           
0.0               1829  613
1.0               1034  466

Chi-square statistic: 16.329
Degrees of freedom: 1
P-value: 0.0001


In [4]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

model = smf.logit(
    "upset ~ spread_magnitude + is_home_underdog + elo_diff + rolling_win_pct_diff + week",
    data=df_model
).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.527301
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                  upset   No. Observations:                 3942
Model:                          Logit   Df Residuals:                     3936
Method:                           MLE   Df Model:                            5
Date:                Sun, 19 Jul 2026   Pseudo R-squ.:                  0.1016
Time:                        19:34:33   Log-Likelihood:                -2078.6
converged:                       True   LL-Null:                       -2313.6
Covariance Type:            nonrobust   LLR p-value:                 2.313e-99
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept                0.1018      0.101      1.008      0.313      -0.096       0.300

In [5]:
print(df_model[["spread_magnitude", "elo_diff", "rolling_win_pct_diff", "is_home_underdog"]].corr())

                      spread_magnitude  elo_diff  rolling_win_pct_diff  \
spread_magnitude              1.000000  0.375749              0.229784   
elo_diff                      0.375749  1.000000              0.607036   
rolling_win_pct_diff          0.229784  0.607036              1.000000   
is_home_underdog             -0.176554 -0.696716             -0.475103   

                      is_home_underdog  
spread_magnitude             -0.176554  
elo_diff                     -0.696716  
rolling_win_pct_diff         -0.475103  
is_home_underdog              1.000000  


In [6]:
import numpy as np
print(df_model[["spread_magnitude"]].assign(elo_diff_abs=df_model["elo_diff"].abs()).corr())

                  spread_magnitude  elo_diff_abs
spread_magnitude          1.000000      0.742461
elo_diff_abs              0.742461      1.000000


In [7]:
model_with_conf = smf.logit(
    "upset ~ spread_magnitude + is_home_underdog + elo_diff + rolling_win_pct_diff + week + C(underdog_conference)",
    data=df_model
).fit()

print(model_with_conf.summary())

Optimization terminated successfully.
         Current function value: 0.525542
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:                  upset   No. Observations:                 3942
Model:                          Logit   Df Residuals:                     3926
Method:                           MLE   Df Model:                           15
Date:                Sun, 19 Jul 2026   Pseudo R-squ.:                  0.1046
Time:                        19:42:17   Log-Likelihood:                -2071.7
converged:                       True   LL-Null:                       -2313.6
Covariance Type:            nonrobust   LLR p-value:                 1.416e-93
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept                                 

In [8]:
from scipy.stats import chi2

lr_stat = 2 * (model_with_conf.llf - model.llf)
df_diff = model_with_conf.df_model - model.df_model
p_value_lr = chi2.sf(lr_stat, df_diff)

print(f"Likelihood ratio statistic: {lr_stat:.3f}")
print(f"Degrees of freedom difference: {df_diff}")
print(f"P-value: {p_value_lr:.4f}")

Likelihood ratio statistic: 13.868
Degrees of freedom difference: 10.0
P-value: 0.1791
